In [ ]:
# ====================
# ИМПОРТ БИБЛИОТЕК
# ====================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn импорты
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import (silhouette_score, davies_bouldin_score, 
                            calinski_harabasz_score, adjusted_rand_score)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors

# ====================
# НАСТРОЙКИ
# ====================

# Настройка стилей графиков
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# Настройка воспроизводимости
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ====================
# ПУТИ К ФАЙЛАМ 
# ====================

# Пути относительно папки, где находится ноутбук
DATA_DIR = 'data/'
ARTIFACTS_DIR = 'artifacts/'
FIGURES_DIR = os.path.join(ARTIFACTS_DIR, 'figures/')
LABELS_DIR = os.path.join(ARTIFACTS_DIR, 'labels/')

# Имена файлов данных
DATASET_FILES = [
    'S07-hw-dataset-01.csv',
    'S07-hw-dataset-02.csv', 
    'S07-hw-dataset-03.csv'
]


# ====================
# ФУНКЦИИ ПРЕПРОЦЕССИНГА
# ====================

def create_preprocessing_pipeline(X):
    
    
    # Определяем типы признаков
    numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
    
    print(f"  Числовые признаки ({len(numeric_features)}): {numeric_features}")
    print(f"  Категориальные признаки ({len(categorical_features)}): {categorical_features}")
    
    # Пайплайн для числовых признаков
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    
    transformers = [
        ('num', numeric_transformer, numeric_features)
    ]
    
    # Если есть категориальные признаки, добавляем их обработку
    if categorical_features:
        from sklearn.preprocessing import OneHotEncoder
        categorical_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ])
        transformers.append(('cat', categorical_transformer, categorical_features))
        print(f"  Добавлен OneHotEncoder для категориальных признаков")
    
    # Создаем ColumnTransformer
    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder='drop'  # удаляем признаки, которые не обрабатываются
    )
    
    return preprocessor

# ====================
# ФУНКЦИИ ДЛЯ КЛАСТЕРИЗАЦИИ
# ====================

def apply_kmeans(X, max_k=20):
    """
    Применение KMeans с подбором оптимального k
    
    Args:
        X: данные для кластеризации
        max_k: максимальное количество кластеров
        
    Returns:
        best_k: оптимальное количество кластеров
        best_labels: метки кластеров для лучшего k
        metrics: метрики для всех k
    """
    
    print(f"\n  Поиск оптимального k для KMeans (2..{max_k})...")
    
    k_range = range(2, max_k + 1)
    silhouette_scores = []
    db_scores = []
    ch_scores = []
    inertia_scores = []
    all_labels = {}
    
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
        labels = kmeans.fit_predict(X)
        all_labels[k] = labels
        
        # Вычисляем метрики только если есть более 1 кластера
        if len(np.unique(labels)) > 1:
            silhouette = silhouette_score(X, labels)
            db = davies_bouldin_score(X, labels)
            ch = calinski_harabasz_score(X, labels)
        else:
            silhouette = -1
            db = np.inf
            ch = -1
            
        silhouette_scores.append(silhouette)
        db_scores.append(db)
        ch_scores.append(ch)
        inertia_scores.append(kmeans.inertia_)
    
    # Находим оптимальное k по Silhouette Score
    best_k_silhouette = k_range[np.argmax(silhouette_scores)]
    best_labels_silhouette = all_labels[best_k_silhouette]
    
    metrics = {
        'k_range': list(k_range),
        'silhouette': silhouette_scores,
        'davies_bouldin': db_scores,
        'calinski_harabasz': ch_scores,
        'inertia': inertia_scores
    }
    
    print(f"  Оптимальное k по Silhouette: {best_k_silhouette}")
    print(f"  Лучший Silhouette Score: {max(silhouette_scores):.3f}")
    
    return best_k_silhouette, best_labels_silhouette, metrics

def apply_dbscan(X, eps_values=None):
    """
    Применение DBSCAN с подбором оптимального eps
    
    Args:
        X: данные для кластеризации
        eps_values: список значений eps для тестирования
        
    Returns:
        best_eps: оптимальное значение eps
        best_labels: метки кластеров для лучшего eps
        metrics: метрики для всех eps
    """
    
    if eps_values is None:
        eps_values = [0.1, 0.3, 0.5, 0.7, 1.0, 1.5]
    
    print(f"\n  Подбор оптимального eps для DBSCAN...")
    print(f"  Тестируемые значения eps: {eps_values}")
    
    results = []
    
    for eps in eps_values:
        dbscan = DBSCAN(eps=eps, min_samples=5)
        labels = dbscan.fit_predict(X)
        
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = list(labels).count(-1)
        noise_ratio = n_noise / len(labels)
        
        # Вычисляем метрики только для нешумовых точек
        non_noise_mask = labels != -1
        if np.sum(non_noise_mask) > 0 and len(np.unique(labels[non_noise_mask])) > 1:
            X_non_noise = X[non_noise_mask]
            labels_non_noise = labels[non_noise_mask]
            
            silhouette = silhouette_score(X_non_noise, labels_non_noise)
            db = davies_bouldin_score(X_non_noise, labels_non_noise)
            ch = calinski_harabasz_score(X_non_noise, labels_non_noise)
        else:
            silhouette = -1
            db = np.inf
            ch = -1
        
        results.append({
            'eps': eps,
            'n_clusters': n_clusters,
            'noise_ratio': noise_ratio,
            'silhouette': silhouette,
            'davies_bouldin': db,
            'calinski_harabasz': ch,
            'labels': labels
        })
    
    # Находим лучший eps по Silhouette Score
    valid_results = [r for r in results if r['silhouette'] > -1]
    if valid_results:
        best_result = max(valid_results, key=lambda x: x['silhouette'])
        best_eps = best_result['eps']
        best_labels = best_result['labels']
    else:
        best_eps = eps_values[0]
        best_labels = results[0]['labels']
    
    print(f"  Оптимальный eps: {best_eps}")
    if valid_results:
        print(f"  Лучший Silhouette Score: {best_result['silhouette']:.3f}")
        print(f"  Количество кластеров: {best_result['n_clusters']}")
        print(f"  Доля шума: {best_result['noise_ratio']:.2%}")
    
    return best_eps, best_labels, results

# ====================
# ФУНКЦИИ ВИЗУАЛИЗАЦИИ
# ====================

def plot_kmeans_metrics(metrics, dataset_name, save_path=None):
    """
    Визуализация метрик для KMeans
    """
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # График 1: Silhouette Score
    axes[0, 0].plot(metrics['k_range'], metrics['silhouette'], 'bo-', linewidth=2)
    axes[0, 0].set_xlabel('Количество кластеров (k)', fontsize=12)
    axes[0, 0].set_ylabel('Silhouette Score', fontsize=12)
    axes[0, 0].set_title('Silhouette Score vs k', fontsize=14)
    axes[0, 0].grid(True, alpha=0.3)
    best_k_idx = np.argmax(metrics['silhouette'])
    axes[0, 0].plot(metrics['k_range'][best_k_idx], metrics['silhouette'][best_k_idx], 
                   'ro', markersize=10, label=f'Лучший k={metrics["k_range"][best_k_idx]}')
    axes[0, 0].legend()
    
    # График 2: Davies-Bouldin Index
    axes[0, 1].plot(metrics['k_range'], metrics['davies_bouldin'], 'go-', linewidth=2)
    axes[0, 1].set_xlabel('Количество кластеров (k)', fontsize=12)
    axes[0, 1].set_ylabel('Davies-Bouldin Index', fontsize=12)
    axes[0, 1].set_title('Davies-Bouldin Index vs k', fontsize=14)
    axes[0, 1].grid(True, alpha=0.3)
    best_k_idx_db = np.argmin(metrics['davies_bouldin'])
    axes[0, 1].plot(metrics['k_range'][best_k_idx_db], metrics['davies_bouldin'][best_k_idx_db], 
                   'ro', markersize=10, label=f'Лучший k={metrics["k_range"][best_k_idx_db]}')
    axes[0, 1].legend()
    
    # График 3: Calinski-Harabasz Index
    axes[1, 0].plot(metrics['k_range'], metrics['calinski_harabasz'], 'mo-', linewidth=2)
    axes[1, 0].set_xlabel('Количество кластеров (k)', fontsize=12)
    axes[1, 0].set_ylabel('Calinski-Harabasz Index', fontsize=12)
    axes[1, 0].set_title('Calinski-Harabasz Index vs k', fontsize=14)
    axes[1, 0].grid(True, alpha=0.3)
    best_k_idx_ch = np.argmax(metrics['calinski_harabasz'])
    axes[1, 0].plot(metrics['k_range'][best_k_idx_ch], metrics['calinski_harabasz'][best_k_idx_ch], 
                   'ro', markersize=10, label=f'Лучший k={metrics["k_range"][best_k_idx_ch]}')
    axes[1, 0].legend()
    
    # График 4: Inertia (Elbow Method)
    axes[1, 1].plot(metrics['k_range'], metrics['inertia'], 'co-', linewidth=2)
    axes[1, 1].set_xlabel('Количество кластеров (k)', fontsize=12)
    axes[1, 1].set_ylabel('Inertia', fontsize=12)
    axes[1, 1].set_title('Метод локтя (Elbow Method)', fontsize=14)
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.suptitle(f'KMeans: Подбор параметров для {dataset_name}', fontsize=16)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  График сохранен: {save_path}")
    
    plt.show()

def plot_dbscan_results(dbscan_results, dataset_name, save_path=None):
    """
    Визуализация результатов DBSCAN
    """
    eps_values = [r['eps'] for r in dbscan_results]
    silhouette_scores = [r['silhouette'] for r in dbscan_results]
    n_clusters = [r['n_clusters'] for r in dbscan_results]
    noise_ratios = [r['noise_ratio'] for r in dbscan_results]
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # График 1: Silhouette Score vs eps
    axes[0, 0].plot(eps_values, silhouette_scores, 'bo-', linewidth=2)
    axes[0, 0].set_xlabel('eps', fontsize=12)
    axes[0, 0].set_ylabel('Silhouette Score', fontsize=12)
    axes[0, 0].set_title('Silhouette Score vs eps', fontsize=14)
    axes[0, 0].grid(True, alpha=0.3)
    
    # График 2: Количество кластеров vs eps
    axes[0, 1].plot(eps_values, n_clusters, 'go-', linewidth=2)
    axes[0, 1].set_xlabel('eps', fontsize=12)
    axes[0, 1].set_ylabel('Количество кластеров', fontsize=12)
    axes[0, 1].set_title('Количество кластеров vs eps', fontsize=14)
    axes[0, 1].grid(True, alpha=0.3)
    
    # График 3: Доля шума vs eps
    axes[1, 0].plot(eps_values, noise_ratios, 'ro-', linewidth=2)
    axes[1, 0].set_xlabel('eps', fontsize=12)
    axes[1, 0].set_ylabel('Доля шума', fontsize=12)
    axes[1, 0].set_title('Доля шума vs eps', fontsize=14)
    axes[1, 0].grid(True, alpha=0.3)
    
    # График 4: K-distance graph (для подбора eps)
    # Этот график нужно строить отдельно, так как он требует других вычислений
    axes[1, 1].text(0.5, 0.5, 'Для построения k-distance графика\nнужны исходные данные',
                   horizontalalignment='center', verticalalignment='center',
                   transform=axes[1, 1].transAxes, fontsize=12)
    axes[1, 1].set_title('K-distance график', fontsize=14)
    axes[1, 1].axis('off')
    
    plt.suptitle(f'DBSCAN: Подбор параметров для {dataset_name}', fontsize=16)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  График сохранен: {save_path}")
    
    plt.show()

def plot_pca_clusters(X, labels, dataset_name, algorithm_name, save_path=None):
    """
    Визуализация кластеров в 2D PCA пространстве
    """
    # Применяем PCA для уменьшения размерности до 2D
    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    X_pca = pca.fit_transform(X)
    
    # Вычисляем процент объясненной дисперсии
    explained_variance = pca.explained_variance_ratio_.sum()
    
    plt.figure(figsize=(10, 8))
    
    # Если есть шумовые точки (label = -1), рисуем их отдельно
    if -1 in labels:
        # Шумовые точки
        noise_mask = labels == -1
        plt.scatter(X_pca[noise_mask, 0], X_pca[noise_mask, 1], 
                   c='gray', alpha=0.3, s=20, label='Шум (outliers)', edgecolors='black', linewidth=0.5)
        
        # Нешумовые точки
        non_noise_mask = ~noise_mask
        scatter = plt.scatter(X_pca[non_noise_mask, 0], X_pca[non_noise_mask, 1], 
                             c=labels[non_noise_mask], cmap='tab20', alpha=0.8, 
                             s=50, edgecolor='k', linewidth=0.5)
    else:
        # Все точки - кластеры
        scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], 
                             c=labels, cmap='tab20', alpha=0.8, 
                             s=50, edgecolor='k', linewidth=0.5)
    
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} дисперсии)', fontsize=12)
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} дисперсии)', fontsize=12)
    plt.title(f'PCA визуализация: {algorithm_name} на {dataset_name}\n'
              f'Объясненная дисперсия: {explained_variance:.1%}', fontsize=14)
    
    # Добавляем цветовую шкалу (если это не шум)
    if -1 not in labels:
        plt.colorbar(scatter, label='Метка кластера')
    
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  PCA график сохранен: {save_path}")
    
    plt.show()

# ====================
# ФУНКЦИИ ДЛЯ ПРОВЕРКИ УСТОЙЧИВОСТИ
# ====================

def check_kmeans_stability(X, n_clusters, n_runs=5):
    """
    Проверка устойчивости KMeans
    
    Args:
        X: данные для кластеризации
        n_clusters: количество кластеров
        n_runs: количество запусков
        
    Returns:
        ari_scores: список ARI между запусками
        average_ari: средний ARI
    """
    print(f"\n  Проверка устойчивости KMeans (k={n_clusters})...")
    
    all_labels = []
    random_states = np.random.randint(0, 10000, size=n_runs)
    
    for i, rs in enumerate(random_states):
        kmeans = KMeans(n_clusters=n_clusters, random_state=rs, n_init=10)
        labels = kmeans.fit_predict(X)
        all_labels.append(labels)
        print(f"    Запуск {i+1} с random_state={rs}")
    
    # Вычисляем Adjusted Rand Index между всеми парами запусков
    ari_scores = []
    for i in range(n_runs):
        for j in range(i+1, n_runs):
            ari = adjusted_rand_score(all_labels[i], all_labels[j])
            ari_scores.append(ari)
    
    average_ari = np.mean(ari_scores)
    print(f"  Средний ARI между запусками: {average_ari:.3f}")
    print(f"  Минимальный ARI: {np.min(ari_scores):.3f}")
    print(f"  Максимальный ARI: {np.max(ari_scores):.3f}")
    
    return ari_scores, average_ari

# ====================
# ОСНОВНОЙ ПРОЦЕСС АНАЛИЗА
# ====================

def analyze_dataset(dataset_file, dataset_idx):
    """
    Полный анализ одного датасета
    
    Args:
        dataset_file: имя CSV файла
        dataset_idx: индекс датасета (1, 2, 3)
        
    Returns:
        results: словарь с результатами анализа
    """
    print("\n" + "="*60)
    print(f"АНАЛИЗ ДАТАСЕТА {dataset_idx}: {dataset_file}")
    print("="*60)
    
    # Загрузка данных (относительный путь)
    data_path = os.path.join(DATA_DIR, dataset_file)
    print(f"\n1. ЗАГРУЗКА ДАННЫХ из: {data_path}")
    df = pd.read_csv(data_path)
    print(f"   Размер данных: {df.shape[0]} строк, {df.shape[1]} столбцов")
    print(f"   Столбцы: {list(df.columns)}")
    
    # Отделяем sample_id и признаки
    sample_ids = df['sample_id']
    X = df.drop('sample_id', axis=1)
    
    # Базовый EDA
    print("\n2. БАЗОВЫЙ АНАЛИЗ (EDA)")
    print(f"   Типы признаков:")
    print(f"   {X.dtypes.value_counts().to_dict()}")
    
    # Проверка пропусков
    missing_values = X.isnull().sum()
    if missing_values.sum() > 0:
        print(f"   Найдены пропуски:")
        for col, count in missing_values[missing_values > 0].items():
            print(f"     {col}: {count} пропусков ({count/len(X):.1%})")
    else:
        print(f"   Пропуски не обнаружены")
    
    # Базовые статистики
    print(f"\n   Базовые статистики числовых признаков:")
    numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns
    if len(numeric_cols) > 0:
        print(f"   Средние значения: {X[numeric_cols].mean().round(2).to_dict()}")
        print(f"   Стандартные отклонения: {X[numeric_cols].std().round(2).to_dict()}")
    
    # ПРЕПРОЦЕССИНГ
    print("\n3. ПРЕПРОЦЕССИНГ")
    preprocessor = create_preprocessing_pipeline(X)
    X_processed = preprocessor.fit_transform(X)
    print(f"   Размер после препроцессинга: {X_processed.shape}")
    print(f"   Использованные трансформеры: {preprocessor.named_transformers_.keys()}")
    
    # КЛАСТЕРИЗАЦИЯ KMeans
    print("\n4. КЛАСТЕРИЗАЦИЯ KMeans")
    best_k_kmeans, labels_kmeans, kmeans_metrics = apply_kmeans(X_processed, max_k=15)
    
    # Визуализация метрик KMeans (относительный путь)
    kmeans_plot_path = os.path.join(FIGURES_DIR, f'kmeans_metrics_ds{dataset_idx}.png')
    plot_kmeans_metrics(kmeans_metrics, f"Dataset {dataset_idx}", save_path=kmeans_plot_path)
    
    # КЛАСТЕРИЗАЦИЯ DBSCAN
    print("\n5. КЛАСТЕРИЗАЦИЯ DBSCAN")
    best_eps, labels_dbscan, dbscan_results = apply_dbscan(X_processed)
    
    # Визуализация результатов DBSCAN (относительный путь)
    dbscan_plot_path = os.path.join(FIGURES_DIR, f'dbscan_metrics_ds{dataset_idx}.png')
    plot_dbscan_results(dbscan_results, f"Dataset {dataset_idx}", save_path=dbscan_plot_path)
    
    # ВЫЧИСЛЕНИЕ МЕТРИК
    print("\n6. ВЫЧИСЛЕНИЕ МЕТРИК КАЧЕСТВА")
    
    # Метрики для KMeans
    sil_kmeans = silhouette_score(X_processed, labels_kmeans)
    db_kmeans = davies_bouldin_score(X_processed, labels_kmeans)
    ch_kmeans = calinski_harabasz_score(X_processed, labels_kmeans)
    
    print(f"   KMeans (k={best_k_kmeans}):")
    print(f"     Silhouette Score: {sil_kmeans:.3f}")
    print(f"     Davies-Bouldin Index: {db_kmeans:.3f}")
    print(f"     Calinski-Harabasz Index: {ch_kmeans:.3f}")
    
    # Метрики для DBSCAN
    if -1 in labels_dbscan:
        # Для DBSCAN с шумом вычисляем метрики только на нешумовых точках
        non_noise_mask = labels_dbscan != -1
        if np.sum(non_noise_mask) > 0 and len(np.unique(labels_dbscan[non_noise_mask])) > 1:
            X_non_noise = X_processed[non_noise_mask]
            labels_non_noise = labels_dbscan[non_noise_mask]
            
            sil_dbscan = silhouette_score(X_non_noise, labels_non_noise)
            db_dbscan = davies_bouldin_score(X_non_noise, labels_non_noise)
            ch_dbscan = calinski_harabasz_score(X_non_noise, labels_non_noise)
        else:
            sil_dbscan = -1
            db_dbscan = np.inf
            ch_dbscan = -1
    else:
        sil_dbscan = silhouette_score(X_processed, labels_dbscan)
        db_dbscan = davies_bouldin_score(X_processed, labels_dbscan)
        ch_dbscan = calinski_harabasz_score(X_processed, labels_dbscan)
    
    noise_ratio_dbscan = list(labels_dbscan).count(-1) / len(labels_dbscan) if -1 in labels_dbscan else 0
    
    print(f"   DBSCAN (eps={best_eps}):")
    if sil_dbscan != -1:
        print(f"     Silhouette Score: {sil_dbscan:.3f}")
        print(f"     Davies-Bouldin Index: {db_dbscan:.3f}")
        print(f"     Calinski-Harabasz Index: {ch_dbscan:.3f}")
    print(f"     Доля шума: {noise_ratio_dbscan:.2%}")
    
    # ВИЗУАЛИЗАЦИЯ PCA
    print("\n7. ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ")
    
    # Выбираем лучший алгоритм по Silhouette Score
    if sil_kmeans >= sil_dbscan:
        best_algorithm = 'KMeans'
        best_labels = labels_kmeans
        best_params = {'k': best_k_kmeans}
        best_silhouette = sil_kmeans
    else:
        best_algorithm = 'DBSCAN'
        best_labels = labels_dbscan
        best_params = {'eps': best_eps, 'min_samples': 5}
        best_silhouette = sil_dbscan
    
    print(f"   Лучший алгоритм: {best_algorithm}")
    print(f"   Лучший Silhouette Score: {best_silhouette:.3f}")
    
    # PCA визуализация для лучшего алгоритма (относительный путь)
    pca_plot_path = os.path.join(FIGURES_DIR, f'pca_best_ds{dataset_idx}.png')
    plot_pca_clusters(X_processed, best_labels, f"Dataset {dataset_idx}", best_algorithm, save_path=pca_plot_path)
    
    # PCA визуализация для KMeans (для сравнения, относительный путь)
    pca_kmeans_path = os.path.join(FIGURES_DIR, f'pca_kmeans_ds{dataset_idx}.png')
    plot_pca_clusters(X_processed, labels_kmeans, f"Dataset {dataset_idx}", "KMeans", save_path=pca_kmeans_path)
    
    # СОХРАНЕНИЕ МЕТОК КЛАСТЕРОВ (относительный путь)
    print("\n8. СОХРАНЕНИЕ РЕЗУЛЬТАТОВ")
    
    # Сохраняем метки лучшего алгоритма
    labels_df = pd.DataFrame({
        'sample_id': sample_ids,
        'cluster_label': best_labels
    })
    
    labels_file_path = os.path.join(LABELS_DIR, f'labels_hw07_ds{dataset_idx}.csv')
    labels_df.to_csv(labels_file_path, index=False)
    print(f"   Метки кластеров сохранены в: {labels_file_path}")
    print(f"   Уникальные метки: {sorted(np.unique(best_labels).tolist())}")
    
    # Подготавливаем результаты
    results = {
        'dataset_name': dataset_file,
        'dataset_shape': df.shape,
        'missing_values': int(missing_values.sum()),
        'preprocessor': {
            'transformers': list(preprocessor.named_transformers_.keys()),
            'output_shape': X_processed.shape
        },
        'kmeans': {
            'best_k': int(best_k_kmeans),
            'silhouette': float(sil_kmeans),
            'davies_bouldin': float(db_kmeans),
            'calinski_harabasz': float(ch_kmeans),
            'labels_count': len(np.unique(labels_kmeans))
        },
        'dbscan': {
            'best_eps': float(best_eps),
            'silhouette': float(sil_dbscan) if sil_dbscan != -1 else None,
            'davies_bouldin': float(db_dbscan) if db_dbscan != np.inf else None,
            'calinski_harabasz': float(ch_dbscan) if ch_dbscan != -1 else None,
            'noise_ratio': float(noise_ratio_dbscan),
            'n_clusters': len(np.unique(labels_dbscan)) - (1 if -1 in labels_dbscan else 0),
            'labels_count': len(np.unique(labels_dbscan))
        },
        'best_algorithm': {
            'name': best_algorithm,
            'params': best_params,
            'silhouette': float(best_silhouette)
        },
        'artifacts': {
            'kmeans_plot': kmeans_plot_path,
            'dbscan_plot': dbscan_plot_path,
            'pca_best_plot': pca_plot_path,
            'pca_kmeans_plot': pca_kmeans_path,
            'labels_file': labels_file_path
        }
    }
    
    print("\n" + "="*60)
    print(f"АНАЛИЗ ДАТАСЕТА {dataset_idx} ЗАВЕРШЕН")
    print("="*60)
    
    return results

# ====================
# ГЛАВНАЯ ФУНКЦИЯ
# ====================

def main():
    """
    Главная функция выполнения домашнего задания
    """
    print("="*60)
    print("ДОМАШНЕЕ ЗАДАНИЕ HW07: КЛАСТЕРИЗАЦИЯ")
    print("="*60)
    
    # Создание структуры папок
    #create_directory_structure()
    
    # Проверяем наличие файлов данных
    print("\nПРОВЕРКА НАЛИЧИЯ ДАТАСЕТОВ:")
    for i, dataset_file in enumerate(DATASET_FILES, 1):
        data_path = os.path.join(DATA_DIR, dataset_file)
        if os.path.exists(data_path):
            print(f"  Dataset {i}: {data_path} ✓")
        else:
            print(f"  Dataset {i}: {data_path} ✗ (ФАЙЛ НЕ НАЙДЕН!)")
            print(f"    Убедитесь, что файлы находятся в папке {DATA_DIR}")
            return
    
    # Анализ каждого датасета
    all_results = {}
    
    for i, dataset_file in enumerate(DATASET_FILES, 1):
        results = analyze_dataset(dataset_file, i)
        all_results[f'dataset_{i}'] = results
    
    # Проверка устойчивости (для первого датасета)
    print("\n" + "="*60)
    print("ПРОВЕРКА УСТОЙЧИВОСТИ KMeans (для Dataset 1)")
    print("="*60)
    
    # Загружаем первый датасет для проверки устойчивости
    first_dataset_file = DATASET_FILES[0]
    data_path = os.path.join(DATA_DIR, first_dataset_file)
    df_stability = pd.read_csv(data_path)
    X_stability = df_stability.drop('sample_id', axis=1)
    
    # Создаем препроцессинг
    preprocessor_stability = create_preprocessing_pipeline(X_stability)
    X_stability_processed = preprocessor_stability.fit_transform(X_stability)
    
    # Оптимальное k для первого датасета
    best_k_stability = all_results['dataset_1']['kmeans']['best_k']
    
    # Проверяем устойчивость
    ari_scores, avg_ari = check_kmeans_stability(X_stability_processed, best_k_stability, n_runs=5)
    
    # Визуализация устойчивости (относительный путь)
    plt.figure(figsize=(10, 6))
    positions = np.arange(len(ari_scores))
    plt.bar(positions, ari_scores, alpha=0.7, color='steelblue')
    plt.axhline(y=avg_ari, color='red', linestyle='--', linewidth=2, 
                label=f'Средний ARI: {avg_ari:.3f}')
    plt.title(f'Проверка устойчивости KMeans\nDataset 1, k={best_k_stability}', fontsize=14)
    plt.xlabel('Пары запусков', fontsize=12)
    plt.ylabel('Adjusted Rand Index (ARI)', fontsize=12)
    plt.xticks(positions, [f'{i+1}' for i in range(len(ari_scores))])
    plt.legend()
    plt.grid(True, alpha=0.3, axis='y')
    plt.ylim(0, 1.1)
    
    stability_plot_path = os.path.join(FIGURES_DIR, 'stability_check.png')
    plt.savefig(stability_plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  График устойчивости сохранен: {stability_plot_path}")
    
    # Добавляем результаты устойчивости к общим результатам
    all_results['stability_check'] = {
        'dataset': first_dataset_file,
        'best_k': int(best_k_stability),
        'average_ari': float(avg_ari),
        'min_ari': float(np.min(ari_scores)),
        'max_ari': float(np.max(ari_scores)),
        'ari_scores': [float(score) for score in ari_scores],
        'plot_path': stability_plot_path
    }
    
    # Шаг 5: Сохранение сводных результатов (относительные пути)
    print("\n" + "="*60)
    print("СОХРАНЕНИЕ СВОДНЫХ РЕЗУЛЬТАТОВ")
    print("="*60)
    
    # Сохраняем метрики
    metrics_summary = {}
    for key, result in all_results.items():
        if key.startswith('dataset_'):
            dataset_num = key.split('_')[1]
            metrics_summary[f'dataset_{dataset_num}'] = {
                'kmeans': result['kmeans'],
                'dbscan': result['dbscan'],
                'best_algorithm': result['best_algorithm']
            }
    
    metrics_file = os.path.join(ARTIFACTS_DIR, 'metrics_summary.json')
    with open(metrics_file, 'w', encoding='utf-8') as f:
        json.dump(metrics_summary, f, indent=2, ensure_ascii=False)
    print(f"  Метрики сохранены в: {metrics_file}")
    
    # Сохраняем лучшие конфигурации
    best_configs = {}
    for key, result in all_results.items():
        if key.startswith('dataset_'):
            dataset_num = key.split('_')[1]
            best_configs[f'dataset_{dataset_num}'] = {
                'best_algorithm': result['best_algorithm']['name'],
                'params': result['best_algorithm']['params'],
                'silhouette_score': result['best_algorithm']['silhouette']
            }
    
    configs_file = os.path.join(ARTIFACTS_DIR, 'best_configs.json')
    with open(configs_file, 'w', encoding='utf-8') as f:
        json.dump(best_configs, f, indent=2, ensure_ascii=False)
    print(f"  Лучшие конфигурации сохранены в: {configs_file}")
    
    # Шаг 6: Вывод итоговой информации
    print("\n" + "="*60)
    print("ИТОГОВЫЕ РЕЗУЛЬТАТЫ")
    print("="*60)
    
    print("\nЛучшие алгоритмы для каждого датасета:")
    for key, result in all_results.items():
        if key.startswith('dataset_'):
            dataset_num = key.split('_')[1]
            best_algo = result['best_algorithm']['name']
            best_params = result['best_algorithm']['params']
            silhouette = result['best_algorithm']['silhouette']
            
            print(f"\nDataset {dataset_num}:")
            print(f"  Лучший алгоритм: {best_algo}")
            print(f"  Параметры: {best_params}")
            print(f"  Silhouette Score: {silhouette:.3f}")
            
            if best_algo == 'DBSCAN':
                noise_ratio = result['dbscan']['noise_ratio']
                print(f"  Доля шума: {noise_ratio:.2%}")
    
    
    # Считаем количество файлов в figures и labels
    figures_count = len([f for f in os.listdir(FIGURES_DIR) if f.endswith('.png')])
    labels_count = len([f for f in os.listdir(LABELS_DIR) if f.endswith('.csv')])
    
    print(f"    ├── figures/ - графики ({figures_count} файлов)")
    print(f"    └── labels/ - метки кластеров ({labels_count} файлов)")
    
    print("\n" + "="*60)
    print("ВЫПОЛНЕНИЕ ЗАДАНИЯ ЗАВЕРШЕНО УСПЕШНО!")
    print("="*60)
    
    return all_results

# ====================
# ЗАПУСК ПРОГРАММЫ
# ====================

if __name__ == "__main__":
    # Выполняем основной код
    results = main()

ДОМАШНЕЕ ЗАДАНИЕ HW07: КЛАСТЕРИЗАЦИЯ

ПРОВЕРКА НАЛИЧИЯ ДАТАСЕТОВ:
  Dataset 1: data/S07-hw-dataset-01.csv ✗ (ФАЙЛ НЕ НАЙДЕН!)
    Убедитесь, что файлы находятся в папке data/
